# EV Adoption, Clean Energy, and Transport Emissions
### DATA 5310 Final Project

Team: Devlin, Naomi, Prithika, Njenga

Our question: Do countries with higher EV adoption have lower transport emissions, and does it matter how clean their electricity is?

In this notebook we combine the four datasets each of us cleaned separately into one dataset that we'll use for our visualizations. All of the data comes from Our World in Data. We focused on 10 countries: Norway, China, the United States, Germany, the United Kingdom, Brazil, Thailand, Japan, India, and South Africa. We picked these to get a mix of regions and a range of EV adoption levels.

| Dataset | Column(s) | Unit | Years |
|---|---|---|---|
| Share of new cars sold that are electric | ev_share | % of new car sales | 2010 to 2025 |
| CO2 emissions from transport per capita | transport_co2 | kg per person | 2010 to 2023 |
| Fossil fuel consumption per capita | gas, oil, coal, total, and fuel shares | kWh per person and % | 2010 to 2025 |
| Share of electricity from renewables | renewable_share | % of electricity | 2010 to 2025 |


Since our research question focuses on comparing countries, we wanted to select a diverse group of countries. Rather than focusing exclusively on countries with the highest adoption rates, we wanted to include countries that could provide interesting comparisons and help us understand how EV adoption relates to overall environmental impact.

We selected Norway because it is a global leader in EV adoption. China was an important addition because it is a major manufacturer of EVs and plays a significant role in the global EV market. Including China also allows us to consider the environmental impact of EV manufacturing alongside the cleanliness of the electricity used to charge these vehicles.

We included the United States because it provides an interesting comparison to countries with higher adoption rates. EVs are generally more expensive here, and consumer preferences for larger vehicles such as trucks and SUVs introduce additional considerations when examining transportation emissions. Germany and the United Kingdom were selected to represent major EV markets in Europe.

To expand our geographic representation, we included Brazil as a leading EV market in Latin America, Thailand to represent the growing EV market in Southeast Asia, and South Africa to represent one of Africa's leading EV markets by sales volume. Finally, we selected Japan and India because they offer interesting contrasts. Japan has historically emphasized hybrid vehicles through manufacturers such as Toyota and Honda, while India has relatively low EV adoption despite its large population and growing automotive market.

Our final list consists of ten countries: Norway, China, United States, Germany, United Kingdom, Brazil, Thailand, South Africa, Japan, and India.

We recognize that our selection still leans toward more developed economies, partly because EV adoption and available data are more limited in many developing countries. However, we tried to include geographic diversity and a range of EV adoption levels, and automotive market characteristics. This should give us a more meaningful comparison than simply selecting the ten countries with the highest EV adoption rates.

### Step 1: Load the data

Each of us saved our cleaned dataset as a CSV. Here we load all four.

In [ ]:
import pandas as pd

In [ ]:
# load the required datasets
emission = pd.read_csv("Emission_transport_cleaned.csv")
renewable = pd.read_csv("renewables_cleaned_1.csv")
fossil_fuel = pd.read_csv("fossil_fuel_data_cleaned.csv")
ev_sales = pd.read_csv("ev_share_clean.csv")


In [ ]:
fossil_fuel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   entity      160 non-null    object 
 1   code        160 non-null    object 
 2   year        160 non-null    int64  
 3   gas_kwh     160 non-null    float64
 4   oil_kwh     160 non-null    float64
 5   coal_kwh    160 non-null    float64
 6   total_kwh   160 non-null    float64
 7   gas_share   160 non-null    float64
 8   oil_share   160 non-null    float64
 9   coal_share  160 non-null    float64
dtypes: float64(7), int64(1), object(2)
memory usage: 12.6+ KB


In [ ]:
renewable.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 465 entries, 0 to 464
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  465 non-null    int64  
 1   Entity      465 non-null    object 
 2   Code        465 non-null    object 
 3   Year        465 non-null    int64  
 4   Renewables  465 non-null    float64
dtypes: float64(1), int64(2), object(2)
memory usage: 18.3+ KB


In [ ]:
emission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Entity                    140 non-null    object 
 1   Code                      140 non-null    object 
 2   Year                      140 non-null    int64  
 3   Emissions from transport  140 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 4.5+ KB


In [ ]:
ev_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157 entries, 0 to 156
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Entity    157 non-null    object 
 1   Code      157 non-null    object 
 2   Year      157 non-null    int64  
 3   ev_share  157 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 5.0+ KB


In [ ]:
renewable["Entity"].unique()

array(['Brazil', 'China', 'Germany', 'India', 'Japan', 'Norway',
       'South Africa', 'Thailand', 'United Kingdom', 'United States'],
      dtype=object)

### Step 2: Fix column names

A few things had to be fixed before we could merge. The fossil fuel data used lowercase column names (entity, code, year), so we changed them to match the other datasets. The renewables file had an extra unnamed column from being saved with its index, so we dropped it. We also shortened some of the long column names to make them easier to work with.

In [ ]:
# Make fossil fuel columns match the others
fossil_fuel = fossil_fuel.rename(columns={"entity": "Entity", "code": "Code", "year": "Year"})

# Drop the extra index column from renewables
renewable = renewable.drop(columns=["Unnamed: 0"])

# Rename value columns so they're short and clear
renewable = renewable.rename(columns={"Renewables": "renewable_share"})
emission = emission.rename(columns={"Emissions from transport": "transport_co2"})

### Step 3: Merge

We merged the four datasets on Entity, Code, and Year using an outer merge. We went with outer because the datasets don't all cover the same years, and we didn't want to lose any rows. After merging, we kept only 2010 to 2025 and sorted by country and year.

In [ ]:
# Merge all four datasets, keeping every row
merged = ev_sales.merge(emission, on=["Entity", "Code", "Year"], how="outer")
merged = merged.merge(fossil_fuel, on=["Entity", "Code", "Year"], how="outer")
merged = merged.merge(renewable, on=["Entity", "Code", "Year"], how="outer")

# Keep only 2010 to 2025
merged = merged[(merged["Year"] >= 2010) & (merged["Year"] <= 2025)]
merged = merged.sort_values(["Entity", "Year"]).reset_index(drop=True)

merged.shape

(160, 13)

### Step 4: Check the merged data

The merged data has 160 rows and 13 columns, which is 10 countries over 16 years. There are some missing values we expected. The transport CO2 data stops at 2023, so 2024 and 2025 are empty for that column. South Africa's EV data starts in 2013, so 2010 to 2012 are empty for ev_share. We kept these rows so the other variables still show the latest years. For charts that use transport CO2, we'll stop at 2023.

In [ ]:
# Check the results
print(merged["Entity"].nunique())
merged.isna().sum()

10


Entity              0
Code                0
Year                0
ev_share            3
transport_co2      20
gas_kwh             0
oil_kwh             0
coal_kwh            0
total_kwh           0
gas_share           0
oil_share           0
coal_share          0
renewable_share     0
dtype: int64

In [ ]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Entity           160 non-null    object 
 1   Code             160 non-null    object 
 2   Year             160 non-null    int64  
 3   ev_share         157 non-null    float64
 4   transport_co2    140 non-null    float64
 5   gas_kwh          160 non-null    float64
 6   oil_kwh          160 non-null    float64
 7   coal_kwh         160 non-null    float64
 8   total_kwh        160 non-null    float64
 9   gas_share        160 non-null    float64
 10  oil_share        160 non-null    float64
 11  coal_share       160 non-null    float64
 12  renewable_share  160 non-null    float64
dtypes: float64(10), int64(1), object(2)
memory usage: 16.4+ KB


### Visualization Brainstorming
Here are the 5 visualizations our group has brainstormed as candidates for plotting our data.

#### Plot 1
Scatter Plot:

For our scatter graph we would want to plot fossil fuel consumption on one axis and EV adoption rates on the other. Both variables would be utilizing the position encoding which is their strongest encoding. This graph would help us understand how much if at all EV adoption is associated with decreasing fossil fuel consumption as a whole.

#### Plot 2
Bubble Plot:

For this graph the X-axis would be EV adoption rate and the Y-axis would be per capita transport CO₂ emissions. Bubble size would be fossil fuel consumption per capita and bubble color would be share of electricity generation from renewables (gradient from dirty to clean). Giving the position encodings to EV adoption rates and transport emissions is important because these are our two main points of interest. Including size for fossil fuel consumption helps us see if countries with higher EV adoption are truly cleaner and color to electricity coming from renewables also helps with the is it cleaner aspect of our question by accounting for pollution that wouldn't be in the transport data despite contributing to it. As far as color choice, we would need to pick a sequential color scheme like blues because this is continuous data. This plot helps us look at the bigger picture and helps us select patterns and points of interest to zoom in on.

#### Plot 3
Facetted Plot:

This facetted plot would first require us to group countries into categories (eg Clean Grid and High EV, Fossil Grid and High EV, Clean Grid and Low EV, Fossil Grid and Low EV). After exploring the share of electricity from renewables for each country more, we would be able to group by parameters we see fit and plot their transport fuel consumption over time. Since this is over time, the strongest chart type would be line graphs. The X-axis would be time while the Y-axis would be transport fuel consumption. We would use the color encoding to distinguish counties in these plots. A good color scheme for this would be a nominal color scheme as this is nominal data.

#### Plot 4
Bar Graph:

For this plot we would use the share of electricity coming from renewable sources for each country in the most recent year. The X-axis would be countries while the Y-axis would be the share of electricity coming from renewables. We would sort the countries from highest to lowest renewable share to make comparisons easier. This helps us see which countries have the cleanest grids.

#### Plot 5
Scatter plot:

For this scatter plot we would look at the share of clean electricity on the X-axis and have EV adoption on the Y-axis to see what countries have the cleanest grids and how that associates with their EV adoption rate. Color would be our third encoding besides position so we can differentiate by color. For this we will use a nominal color scheme as this is nominal data.



